# Extension 02 Compression and tensegrity

In [ ]:
%pip install -q numpy session-py==0.310.0 session-jupyter==0.2.0

# Question

**How does changing the sign of $q$ produce compression gridshells, cable strut domes, and free standing tensegrities?**

```
n42 ─ n43 ─ n44 ─ n45 ─ n46 ─ n47 ─ n48
 │     │     │     │     │     │     │
n35 ─ n36 ─ n37 ─ n38 ─ n39 ─ n40 ─ n41
 │     │     │     │     │     │     │
n28 ─ n29 ─ n30 ─ n31 ─ n32 ─ n33 ─ n34
 │     │     │     │     │     │     │
n21 ─ n22 ─ n23 ─ n24 ─ n25 ─ n26 ─ n27
 │     │     │     │     │     │     │
n14 ─ n15 ─ n16 ─ n17 ─ n18 ─ n19 ─ n20
 │     │     │     │     │     │     │
n7  ─ n8  ─ n9  ─ n10 ─ n11 ─ n12 ─ n13
 │     │     │     │     │     │     │
n0  ─ n1  ─ n2  ─ n3  ─ n4  ─ n5  ─ n6
```

**Given:** bar forces $f = Q\,l$ where positive $q_k \gt 0$ means tension pull and negative $q_k \lt 0$ means compression push.

- **Pure compression vault** ($q_k \lt 0$ everywhere): $D_N = C_N^{T}\,Q\,C_N$ is negative definite, inverting gravity loads ${\color{#2e9e4a} p_z \lt 0}$ into an upward funicular arch shell with load path $\sigma_{FL} = \sum_k |f_k|\,l_k$ and fabrication cut lengths $l_{0,k} = \frac{EA\,l_k}{EA + f_k} \gt l_k$.
- **Mixed cable and strut network** ($q_{\text{cable}} \gt 0$, $q_{\text{strut}} \lt 0$): stable equilibrium holds when $D_N \succ 0$ on anchored domes, or when the $3n \times m$ equilibrium matrix $A_q(x)$ has a self stress vector $q$ and $D = C^{T}\,Q\,C$ has nullity $4$ in 3D for free standing tensegrities:

$$
A_q(x)\,q = 0, \qquad D\,x = \left(C^{T}\,Q\,C\right) x = 0
$$

<div align="center">
  <img src="https://raw.githubusercontent.com/BRG-teaching/Teaching/main/CAS/logos/fd_projection_all_axes.svg"
       style="width:100%; height:auto;"
       alt="Three axis bar projections reversing between tension pull and compression thrust" />
  <p style="text-align:center;">
    <em>Reversing the sign of q flips axial bar pull into compression thrust while keeping the linear matrix structure unchanged.</em>
  </p>
</div>

**Open source references:**

- **ShellFormFinding**, https://github.com/amandaghassaei/ShellFormFinding
- **Force Density Method**, https://github.com/sevamoo/Force_Density_Method
- **Force Density Method API**, https://github.com/alexHaridis/ForceDensityAPI
- **Structure Form Finding**, https://github.com/riverinme/Structure_Form_Finding_HH
- **Muscle**, https://github.com/JonasFeron/Muscle
- **Tensegrity Designer**, https://github.com/hermes98761234/tensegrity-designer
- **Modeling of Tensegrity Structures**, https://github.com/ramaniitrgoyal92/Modeling_of_Tensegrity_Structures_MOTES
- **tensegritypy**, https://github.com/jerryderry/tensegritypy

<hr style="border: none; border-top: 2px solid #555; margin: 2em 0;">

# Funicular Compression Gridshell and Fabrication Cut Lengths

Negative force densities $q_k \lt 0$ under downward gravity ${\color{#2e9e4a} p_z \lt 0}$ produce an upward compression vault whose structural demand is measured by the load path $\sigma_{FL} = \sum_k |f_k|\,l_k$. Because $f_k \lt 0$, each timber lath shortens elastically under thrust and must be cut slightly longer than its equilibrium length:

$$
l_{0,k} = \frac{EA\,l_k}{EA + f_k} \gt l_k
$$

In [1]:
import numpy as np
from session_py import Arrowhead, Color, Line, Matrix, Session
from session_jupyter import show

RED = Color(0.84, 0.16, 0.16, 1.0)
BLUE = Color(0.16, 0.44, 0.86, 1.0)
GREEN = Color(0.18, 0.62, 0.29, 1.0)
GRAY = Color(0.72, 0.72, 0.72, 1.0)

# 6 x 6 grid (49 nodes, 84 bars)
n = 7
gx, gy = np.meshgrid(range(n), range(n))
x0 = gx.flatten().astype(float)
y0 = gy.flatten().astype(float)
z0 = np.zeros(n * n)

edges = [(k, k + 1) for k in range(n * n) if x0[k] < n - 1] + [
    (k, k + n) for k in range(n * n) if y0[k] < n - 1
]
C = np.zeros((len(edges), n * n))
for e, (a, b) in enumerate(edges):
    C[e, a], C[e, b] = 1.0, -1.0

fixed = [0, n - 1, n * (n - 1), n * n - 1]
free = [k for k in range(n * n) if k not in fixed]
C_N, C_F = C[:, free], C[:, fixed]

# stiffer compression force density on free boundary arch ribs (-3.5 kN/m) than interior (-1.0 kN/m)
q_shell = np.full(len(edges), -1.0)
for e, (a, b) in enumerate(edges):
    if (x0[a] in (0.0, 6.0) and x0[b] == x0[a]) or (y0[a] in (0.0, 6.0) and y0[b] == y0[a]):
        q_shell[e] = -3.5
Q_shell = np.diag(q_shell)

p_x = np.zeros(len(free))
p_y = np.zeros(len(free))
p_z = np.full(len(free), -0.30)

x_sh, y_sh, z_sh = x0.copy(), y0.copy(), z0.copy()

# equation 6.32
A = C_N.T @ Q_shell @ C_N
x_sh[free] = np.linalg.solve(A, p_x - C_N.T @ Q_shell @ C_F @ x_sh[fixed])
y_sh[free] = np.linalg.solve(A, p_y - C_N.T @ Q_shell @ C_F @ y_sh[fixed])
z_sh[free] = np.linalg.solve(A, p_z - C_N.T @ Q_shell @ C_F @ z_sh[fixed])

xyz_sh = np.column_stack([x_sh, y_sh, z_sh])
l_sh = np.linalg.norm(C @ xyz_sh, axis=1)
f_sh = q_shell * l_sh
sigma_fl = np.sum(np.abs(f_sh) * l_sh)

# unstretched timber lath cut lengths for EA = 1200 kN
EA_lath = 1200.0
l_0_sh = EA_lath * l_sh / (EA_lath + f_sh)

p_full = np.zeros((n * n, 3))
p_full[free, 2] = p_z
reactions_sh = -(p_full - (C.T @ Q_shell @ C) @ xyz_sh)[fixed]

print("eigenvalues of D_N: max =", round(np.linalg.eigvalsh(A).max(), 3), " (negative definite)")
print("vault crown height z24  =", round(z_sh[24], 3), "m")
print("axial forces min / max  =", round(f_sh.min(), 2), "/", round(f_sh.max(), 2), "kN (all compression)")
print("load path sum(|f|*l)    =", round(sigma_fl, 2), "kN m")
print("max precamber l_0 - l   =", round((l_0_sh - l_sh).max() * 1000.0, 2), "mm")

f_abs = np.abs(f_sh)
colors_sh = [Color(0.65 - 0.65 * fv / f_abs.max(), 0.65 - 0.65 * fv / f_abs.max(), 0.65 - 0.65 * fv / f_abs.max(), 1.0) for fv in f_abs]
session_sh = Session("fdm_ext02_compression_shell")
session_sh.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(xyz_sh.tolist()), colors_sh, [3.0] * len(edges))
for k in range(n * n):
    pt = session_sh.objects.points[k]
    pt.pointcolor = BLUE if k in fixed else RED
    pt.width = 14.0 if k in fixed else 6.5

for idx, k in enumerate(fixed):
    r_vec = reactions_sh[idx] * 0.18
    arrow = Line(xyz_sh[k, 0], xyz_sh[k, 1], xyz_sh[k, 2], xyz_sh[k, 0] + r_vec[0], xyz_sh[k, 1] + r_vec[1], xyz_sh[k, 2] + r_vec[2])
    arrow.linecolor = GREEN
    arrow.width = 3.2
    arrow.arrowhead = Arrowhead.END
    session_sh.add_line(arrow)

show(session_sh, "fdm_ext02_compression_shell")

eigenvalues of D_N: max = -0.275  (negative definite)
vault crown height z24  = 1.507 m
axial forces min / max  = -4.41 / -0.72 kN (all compression)
load path sum(|f|*l)    = 143.36 kN m
max precamber l_0 - l   = 4.65 mm


# Anchored Mixed Tension and Compression Cable Dome

Now we combine positive cable force densities ($q_k \gt 0$) and negative floating strut force densities ($q_k \lt 0$) in a circular cable dome of radius $5\text{ m}$ with $8$ radial sectors:

- $8$ outer perimeter anchors ${\color{#2a6fdb} n_0 \ldots n_7}$ fixed on an undulating ring,
- $8$ lower tension hoop nodes ${\color{#d62828} n_8 \ldots n_{15}}$ carried by diagonal stay cables and linked in a tension hoop ring,
- $8$ upper ridge nodes ${\color{#d62828} n_{16} \ldots n_{23}}$ pushed upward by **floating compression struts** ($q_{\text{strut}} = -1.1\text{ kN/m}$) and tied to a central apex ${\color{#d62828} n_{24}}$.

Because $D_N = C_N^{T}\,Q\,C_N$ remains strictly positive definite ($\lambda_{\min}(D_N) \gt 0$), `# equation 6.32` solves the hybrid cable strut dome in one linear pass.

In [2]:
m_sec = 8
angles = np.linspace(0.0, 2.0 * np.pi, m_sec, endpoint=False)

xyz_dome = np.zeros((3 * m_sec + 1, 3))
xyz_dome[0:m_sec, 0] = 5.0 * np.cos(angles)
xyz_dome[0:m_sec, 1] = 5.0 * np.sin(angles)
xyz_dome[m_sec : 2 * m_sec, 0] = 2.5 * np.cos(angles)
xyz_dome[m_sec : 2 * m_sec, 1] = 2.5 * np.sin(angles)
xyz_dome[2 * m_sec : 3 * m_sec, 0] = 2.5 * np.cos(angles)
xyz_dome[2 * m_sec : 3 * m_sec, 1] = 2.5 * np.sin(angles)

dome_edges = []
q_dome = []

# 1. outer lower diagonal stay cables (+4.0 kN/m) and outer upper ridge cables (+3.2 kN/m)
for i in range(m_sec):
    dome_edges.append((i, m_sec + i))
    q_dome.append(4.0)
    dome_edges.append((i, 2 * m_sec + i))
    q_dome.append(3.2)

# 2. lower tension hoop ring (+4.0 kN/m)
for i in range(m_sec):
    dome_edges.append((m_sec + i, m_sec + (i + 1) % m_sec))
    q_dome.append(4.0)

# 3. floating vertical compression struts (-1.1 kN/m)
strut_indices = []
for i in range(m_sec):
    strut_indices.append(len(dome_edges))
    dome_edges.append((m_sec + i, 2 * m_sec + i))
    q_dome.append(-1.1)

# 4. inner upper radial ridge cables to central apex (+3.2 kN/m)
apex = 3 * m_sec
for i in range(m_sec):
    dome_edges.append((2 * m_sec + i, apex))
    q_dome.append(3.2)

q_dome = np.array(q_dome)
Q_dome = np.diag(q_dome)
C_dome = np.zeros((len(dome_edges), len(xyz_dome)))
for e, (a, b) in enumerate(dome_edges):
    C_dome[e, a], C_dome[e, b] = 1.0, -1.0

fixed_d = list(range(m_sec))
free_d = list(range(m_sec, len(xyz_dome)))
C_Nd, C_Fd = C_dome[:, free_d], C_dome[:, fixed_d]

p_dome = np.zeros((len(xyz_dome), 3))
for i in range(m_sec):
    xyz_dome[i, 2] = 0.9 + 0.45 * np.cos(2.0 * angles[i])
p_dome[m_sec : 2 * m_sec, 2] = -0.55
p_dome[2 * m_sec :, 2] = 0.25

# equation 6.32
A_d = C_Nd.T @ Q_dome @ C_Nd
xyz_dome[free_d, 0] = np.linalg.solve(A_d, p_dome[free_d, 0] - C_Nd.T @ Q_dome @ C_Fd @ xyz_dome[fixed_d, 0])
xyz_dome[free_d, 1] = np.linalg.solve(A_d, p_dome[free_d, 1] - C_Nd.T @ Q_dome @ C_Fd @ xyz_dome[fixed_d, 1])
xyz_dome[free_d, 2] = np.linalg.solve(A_d, p_dome[free_d, 2] - C_Nd.T @ Q_dome @ C_Fd @ xyz_dome[fixed_d, 2])

eig_d = np.linalg.eigvalsh(A_d)
l_dome = np.linalg.norm(C_dome @ xyz_dome, axis=1)
f_dome = q_dome * l_dome

print("min eigenvalue of D_N =", round(eig_d.min(), 4), "(> 0: stable mixed tension compression dome)")
print("strut z bottom -> top =", round(xyz_dome[m_sec, 2], 3), "m ->", round(xyz_dome[2 * m_sec, 2], 3), "m")
print("strut axial force     =", round(f_dome[strut_indices[0]], 3), "kN (compression)")
print("hoop cable force      =", round(f_dome[2 * m_sec], 3), "kN (tension)")

colors_d = []
widths_d = []
for e, fv in enumerate(f_dome):
    if fv < 0.0:
        colors_d.append(BLUE)
        widths_d.append(4.8)
    else:
        shade = 0.60 - 0.60 * fv / f_dome.max()
        colors_d.append(Color(shade, shade, shade, 1.0))
        widths_d.append(2.4)

session_d = Session("fdm_ext02_cable_strut_dome")
session_d.add_network(Matrix.from_rows(C_dome.tolist()), Matrix.from_rows(xyz_dome.tolist()), colors_d, widths_d)
for k in range(len(xyz_dome)):
    pt = session_d.objects.points[k]
    pt.pointcolor = BLUE if k in fixed_d else RED
    pt.width = 13.0 if k in fixed_d else 7.5

show(session_d, "fdm_ext02_cable_strut_dome")

min eigenvalue of D_N = 1.2198 (> 0: stable mixed tension compression dome)
strut z bottom -> top = 0.741 m -> 1.434 m
strut axial force     = -1.336 kN (compression)
hoop cable force      = 10.198 kN (tension)


# Free Standing Tensegrity via SVD Self Stress and Nullspace Projection

With **zero fixed supports** and zero external loads (${\color{#2e9e4a} p = 0}$), self equilibrium requires two simultaneous nullspace conditions:

1. The $3n \times m$ equilibrium matrix $A_q(x) = [C^{T}U;\; C^{T}V;\; C^{T}W]$ must have a self stress mode $A_q(x)\,q = 0$ with $q \gt 0$ on cables and $q \lt 0$ on struts.
2. The $n \times n$ stress matrix $D(q) = C^{T}\,Q\,C$ satisfies $D(q)\,[1, x, y, z] = 0$, so a non coplanar 3D tensegrity requires $\operatorname{rank}(D) = n - 4$ (nullity $4$). Alternating SVD self stress extraction with projection onto the $4$ smallest magnitude eigenvectors of $D(q)$ rotates an untwisted prism seed to the exact $30^\circ$ tensegrity simplex.

In [3]:
# 3 strut tensegrity simplex: 6 nodes (bottom 0,1,2 and top 3,4,5), 12 members (9 cables + 3 struts)
ang3 = np.array([0.0, 2.0 * np.pi / 3.0, 4.0 * np.pi / 3.0])
# start from a non equilibrium seed at 8 degrees twist
seed_twist = np.deg2rad(8.0)
pos = np.zeros((6, 3))
pos[0:3, 0], pos[0:3, 1], pos[0:3, 2] = 2.0 * np.cos(ang3), 2.0 * np.sin(ang3), -1.2
pos[3:6, 0], pos[3:6, 1], pos[3:6, 2] = 2.0 * np.cos(ang3 + seed_twist), 2.0 * np.sin(ang3 + seed_twist), 1.2
pos -= pos.mean(axis=0)
rms0 = np.sqrt(np.mean(np.sum(pos**2, axis=1)))

t_edges = [
    (0, 1), (1, 2), (2, 0),  # bottom cable triangle
    (3, 4), (4, 5), (5, 3),  # top cable triangle
    (0, 3), (1, 4), (2, 5),  # 3 saddle cables
    (0, 4), (1, 5), (2, 3),  # 3 compression struts
]
target_sign = np.array([1.0] * 9 + [-1.0] * 3)

C_t = np.zeros((len(t_edges), 6))
for e, (a, b) in enumerate(t_edges):
    C_t[e, a], C_t[e, b] = 1.0, -1.0


def aq_matrix(xyz):
    """3n x m matrix A_q such that A_q @ q = [C^T Q C x; C^T Q C y; C^T Q C z] = 0."""
    u, v, w = C_t @ xyz[:, 0], C_t @ xyz[:, 1], C_t @ xyz[:, 2]
    return np.vstack([C_t.T @ np.diag(u), C_t.T @ np.diag(v), C_t.T @ np.diag(w)])


for step in range(80):
    Aq = aq_matrix(pos)
    _, s_vals, vt = np.linalg.svd(Aq)
    tol_s = s_vals[-1] + 1e-7 * s_vals[0]
    null_modes = vt[s_vals <= tol_s]
    q_t = np.zeros(len(t_edges))
    for mode in null_modes:
        q_t += np.dot(mode, target_sign) * mode
    q_t /= np.mean(q_t[:9])

    D_t = C_t.T @ np.diag(q_t) @ C_t
    rel_res = np.linalg.norm(D_t @ pos) / (np.linalg.norm(D_t) * np.linalg.norm(pos))
    if rel_res < 1e-11:
        break

    eig_vals, eig_vecs = np.linalg.eigh(D_t)
    idx4 = np.argsort(np.abs(eig_vals))[:4]
    basis4 = eig_vecs[:, idx4]
    pos_next = basis4 @ (basis4.T @ pos)
    pos_next *= rms0 / np.sqrt(np.mean(np.sum(pos_next**2, axis=1)))
    pos = pos_next

phi_bot = np.arctan2(pos[0, 1], pos[0, 0])
phi_top = np.arctan2(pos[3, 1], pos[3, 0])
twist_deg = np.rad2deg((phi_top - phi_bot + np.pi) % (2.0 * np.pi) - np.pi)
eig_final = np.linalg.eigvalsh(C_t.T @ np.diag(q_t) @ C_t)

print("converged in", step + 1, "alternating projection steps")
print("relative self equilibrium residual ||D x|| =", f"{rel_res:.2e}")
print("discovered top to bottom twist angle       =", round(twist_deg, 3), "deg (exact simplex = 30.0 deg)")
print("eigenvalues of D = C^T Q C                 =", np.round(eig_final, 5), "(4 zeros + 2 positive)")
print("force densities q (9 cables > 0, 3 struts < 0):\n", np.round(q_t, 4))

pos_view = pos.copy()
pos_view[:, 2] -= pos_view[:, 2].min()
colors_t = [GRAY if qv > 0 else BLUE for qv in q_t]
widths_t = [2.5 if qv > 0 else 5.5 for qv in q_t]

session_t = Session("fdm_ext02_tensegrity_simplex")
session_t.add_network(Matrix.from_rows(C_t.tolist()), Matrix.from_rows(pos_view.tolist()), colors_t, widths_t)
for k in range(6):
    pt = session_t.objects.points[k]
    pt.pointcolor = RED
    pt.width = 10.0

show(session_t, "fdm_ext02_tensegrity_simplex")

converged in 13 alternating projection steps
relative self equilibrium residual ||D x|| = 7.34e-12
discovered top to bottom twist angle       = 30.0 deg (exact simplex = 30.0 deg)
eigenvalues of D = C^T Q C                 = [-0.       0.       0.       0.       4.82309  4.82309] (4 zeros + 2 positive)
force densities q (9 cables > 0, 3 struts < 0):
 [ 0.8038  0.8038  0.8038  0.8038  0.8038  0.8038  1.3923  1.3923  1.3923
 -1.3923 -1.3923 -1.3923]


# Observation

**Force density unifies funicular compression vaults, hybrid cable domes, and free standing tensegrities under one matrix operator $D = C^{T}\,Q\,C$.** Pure compression shells make $D_N$ negative definite; anchored cable domes balance negative strut $q_k$ inside a positive definite $D_N$; and free standing tensegrities sit in the 4 dimensional nullspace of a positive semidefinite $D$ of rank $n - 4$, discovering the exact $30^\circ$ simplex twist without a single fixed node.